# E4, E6, E7, E8 — Experimentos locales (GPU)

Cuatro intervenciones sobre el modelo, cada una variando **un solo factor** bajo el mismo protocolo de P0 (StratifiedKFold, misma semilla). Aislar un factor por experimento evita **confundir variables**: si cambiáramos dos cosas a la vez, no sabríamos cuál causó el efecto.

- **E4:** cabeza ordinal CORN (¿reduce los errores lejanos?).
- **E6:** backbone de español social RoBERTuito/BETO (¿entiende mejor el registro de TikTok?).
- **E7:** limpieza mínima que conserva emojis (¿ayuda no borrar señal?).
- **E8:** datos sintéticos para las clases raras 4/5 (¿ataca el desbalance?).

> Para RE-EJECUTAR: `driver_local.py all` (E6, E7), `exp_e4.py` (E4), `exp_e8_foldaware.py` (E8). Requieren GPU.

In [1]:
# ============================================================================
# CONFIGURACION COMUN
# ----------------------------------------------------------------------------
# Este bloque prepara el entorno. Se repite en todos los notebooks para que
# cada uno sea autonomo (se pueda abrir y correr por separado).
# ============================================================================
import sys                     # para anadir la carpeta 'src' al path de importacion
import json                    # los resultados de cada experimento se guardan como JSON
from pathlib import Path       # manejo de rutas independiente del sistema operativo

# Anadimos experimentos/src al path para poder importar el codigo compartido
# (metricas ordinales, carga de datos, semillas). Usamos rutas relativas para
# que el notebook funcione sin importar donde este clonado el repositorio.
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np             # calculo numerico (vectores de predicciones)
import pandas as pd            # tablas de resultados legibles
import common as C             # nuestro modulo: metricas ordinales, carga del gold, semilla

# Carpeta donde viven los resultados ya calculados (un JSON por experimento).
R = C.RESULTS

def load(nombre_archivo):
    """Carga un JSON de resultados desde experimentos/resultados/."""
    return json.load(open(R / nombre_archivo))

# --- PATRON DE DOS NIVELES (buena practica de reproducibilidad) ---
# Por defecto RECOMPUTE=False: el notebook CARGA los resultados ya calculados,
# corre en segundos y NO necesita GPU ni Amazon Bedrock. Asi un revisor puede
# abrirlo y ver todo sin infraestructura.
# Si pones RECOMPUTE=True, las celdas marcadas volveran a ENTRENAR/LLAMAR a la nube
# (requiere el venv de ML y/o credenciales de Bedrock; toma minutos a horas).
RECOMPUTE = False

# Fijamos la semilla global para que cualquier calculo aleatorio (p.ej. bootstrap)
# sea reproducible: dos ejecuciones dan el mismo numero.
C.set_all_seeds(C.SEED)
print(f"Entorno listo. Semilla global = {C.SEED}. RECOMPUTE = {RECOMPUTE}.")

Entorno listo. Semilla global = 61298. RECOMPUTE = False.


## E4 — Cabeza ordinal CORN (mejora significativa en MAE)

El modelo original trata las 5 clases como categorías sin orden (softmax). CORN convierte el problema en preguntas acumulativas ("¿supera el nivel k?"), enseñándole el **orden**. La métrica que esto debe mejorar es el **MAE** (distancia media del error), no necesariamente el QWK.

In [2]:
# E4: comparamos la cabeza ordinal CORN contra el softmax del baseline (P0).
e4 = load("e4_corn.json")
p0 = load("p0_stratified.json")

# OJO a la distincion de metricas: 0.656 es el MAE de CORN, NO su QWK (que es 0.414).
print(f"softmax (P0):  MAE = {p0['oof_global']['mae']:.3f}   QWK = {p0['oof_global']['qwk']:.3f}")
print(f"CORN ordinal:  MAE = {e4['corn_metrics']['mae']:.3f}   QWK = {e4['corn_metrics']['qwk']:.3f}")
print()
# La mejora esta en el MAE. El IC de la diferencia de MAE excluye 0 => significativo.
d = e4["delta_mae_neg_corn_minus_base"]
print(f"Reduccion de MAE = {d['mean']:+.3f}  IC[{d['ci_low']:+.3f},{d['ci_high']:+.3f}]  -> IC excluye 0: SIGNIFICATIVO")
print("Interpretacion: CORN se equivoca 'por poco' (predice 4 cuando era 5), no 'por mucho' (predice 1).")

softmax (P0):  MAE = 0.761   QWK = 0.415
CORN ordinal:  MAE = 0.656   QWK = 0.414

Reduccion de MAE = +0.106  IC[+0.043,+0.164]  -> IC excluye 0: SIGNIFICATIVO
Interpretacion: CORN se equivoca 'por poco' (predice 4 cuando era 5), no 'por mucho' (predice 1).


## E6 — Backbone de español social

In [3]:
# E6: mismo pipeline, cambiando SOLO el modelo base.
e6 = load("e6_backbone.json")
pd.DataFrame([[k, round(v["oof_global"]["qwk"], 3)] for k, v in e6.items() if "oof_global" in v],
             columns=["backbone", "QWK"]).sort_values("QWK", ascending=False)

,backbone,QWK
2,robertuito,0.470
0,nlptown-baseline,0.415
1,beto,0.320


**RoBERTuito** (preentrenado en ~500M de tuits en español) obtiene el mayor QWK (0.470). **BETO** (español de Wikipedia/noticias) queda **por debajo** del baseline (0.320). Lección: estar en español no basta; importa el **dominio** (texto social). *Nota honesta:* no calculamos el IC pareado de E6, así que lo reportamos como sugerente, no confirmado.

## E7 — Limpieza mínima vs agresiva (resultado nulo)

In [4]:
# E7: comparamos la limpieza agresiva actual (borra emojis/puntuacion) contra una
# minima que los conserva, con el MISMO modelo. Solo cambia el preprocesamiento.
e7 = load("e7_limpieza.json"); d = e7["delta_qwk_minima_minus_agresiva"]
print(f"limpieza agresiva:  QWK = {e7['text_agresiva']['oof_global']['qwk']:.3f}")
print(f"limpieza minima:    QWK = {e7['text_minima']['oof_global']['qwk']:.3f}")
print(f"Delta (minima - agresiva) = {d['mean']:+.3f}  IC[{d['ci_low']:+.3f},{d['ci_high']:+.3f}]  -> dentro del ruido")

limpieza agresiva:  QWK = 0.443
limpieza minima:    QWK = 0.460
Delta (minima - agresiva) = +0.017  IC[-0.017,+0.051]  -> dentro del ruido


Conservar emojis mejora un poco (+0.017) pero **dentro del ruido**: dirección esperada, no concluyente con este N. Se corrió *antes* que E6 para no confundir "modelo nuevo" con "limpieza nueva".

## E8 — Datos sintéticos para clases raras (significativo, SIN fuga)

Las clases positivas (4 y 5) tienen solo 45 ejemplos. Generamos paráfrasis con un LLM. **Punto crítico de honestidad:** las paráfrasis se generan **solo de los positivos del train de cada fold**, nunca de los de validación, para que el modelo no vea una copia casi idéntica de un ejemplo de test (eso sería *near-duplicate leakage*).

In [5]:
# E8 version FOLD-AWARE (sin fuga). Es la valida; la version con fuga se conserva solo para comparar.
e8 = load("e8_augmentation_foldaware.json"); d = e8["delta_qwk_vs_p0"]
print(f"F1 de clases 4/5:  {e8['f1_clases45_base']:.3f} -> {e8['f1_clases45_aug']:.3f}  (con +{e8['n_synthetic']} sinteticos)")
print(f"Delta QWK vs baseline = {d['mean']:+.3f}  IC[{d['ci_low']:+.3f},{d['ci_high']:+.3f}]  -> IC excluye 0: SIGNIFICATIVO")
print()
# Comparacion honesta con la version que TENIA fuga, para mostrar que el efecto sobrevive.
e8_leak = load("e8_augmentation.json")
print(f"(Comparacion) version CON fuga daba delta +{e8_leak['delta_qwk_vs_p0']['mean']:.3f}; "
      f"al corregir el leakage baja a +{d['mean']:.3f} pero SIGUE significativo.")

F1 de clases 4/5:  0.223 -> 0.262  (con +135 sinteticos)
Delta QWK vs baseline = +0.045  IC[+0.005,+0.087]  -> IC excluye 0: SIGNIFICATIVO

(Comparacion) version CON fuga daba delta +0.057; al corregir el leakage baja a +0.045 pero SIGUE significativo.


## Veredicto de E4-E8
Dos mejoras con IC que excluye 0: **E4** (MAE) y **E8 fold-aware** (QWK y F1 de clases raras). **E6** es sugerente (mayor QWK, sin IC pareado). **E7** dentro del ruido. La combinación RoBERTuito + CORN + augmentation fold-aware es el trabajo futuro natural, idealmente con más datos.